#**BÀI TẬP CHƯƠNG 2**


In [47]:
import sqlite3
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

conn = sqlite3.connect(":memory:")

def query_df(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

def run_script(sql: str) -> None:
    conn.executescript(sql)

def show(title: str, sql: str) -> None:
    display(Markdown(f"### {title}"))
    display(query_df(sql))

#Tạo dữ liệu gốc

In [48]:
run_script('''
DROP TABLE IF EXISTS student;
DROP TABLE IF EXISTS course;

CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
);

CREATE TABLE course (
    id INTEGER,
    course_name TEXT
);

INSERT INTO student (student_id, name, class, course_id, score) VALUES
(1,  'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
(2,  'Tran Thi Lan',      'Kinh Te',  34, 9.2),
(3,  'Pham Van Nam',      'Toan Tin', NULL, 7.9),
(4,  'Le Thanh Huyen',    'Toan Tin', 20, 7.2),
(5,  'Vu Quoc Anh',       'May Tinh', 24, 8.0),
(6,  'Dang Thuy Linh',    'May Tinh', 24, 5.5),
(7,  'Bui Tien Dung',     'Kinh Te',  34, 9.2),
(8,  'Ho Ngoc Mai',       'Toan Tin', 20, 8.8),
(9,  'Duong Huu Phuc',    'Kinh Te',  NULL, 7.2),
(10, 'Cao Thi Hanh',      'May Tinh', NULL, 7.0);

INSERT INTO course (id, course_name) VALUES
(12, 'Giai tich'),
(34, 'Thong ke'),
(26, 'Tin hoc');
''')

show("Bảng student", "SELECT * FROM student ORDER BY student_id;")
show("Bảng course", "SELECT * FROM course ORDER BY id;")

### Bảng student

,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,9.2
2,3,Pham Van Nam,Toan Tin,NaN,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,7.0


### Bảng course

,id,course_name
0,12,Giai tich
1,26,Tin hoc
2,34,Thong ke


# **1.Kết nối hai bảng**



## Tích Descartes
Tích Descartes tạo ra mọi cặp kết hợp giữa từng dòng của `student` và từng dòng của `course`.
Với 10 sinh viên và 3 môn học

In [49]:
show(
    "Tích Descartes giữa student và course",
    '''
    SELECT
        s.student_id,
        s.name,
        s.class,
        s.course_id AS student_course_id,
        s.score,
        c.id AS course_id_from_course,
        c.course_name
    FROM student AS s, course AS c
    ORDER BY s.student_id, c.id;
    '''
)

### Tích Descartes giữa student và course

,student_id,name,class,student_course_id,score,course_id_from_course,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
8,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


### Kết luận về Tích Descartes từ kết quả
- Số lượng bản ghi: Trả về đúng 30 dòng (10 sinh viên × 3 môn học).

- Bản chất ghép cặp: Mỗi sinh viên được tự động ghép với toàn bộ các môn học vì câu lệnh không có điều kiện kết nối (như WHERE hoặc JOIN ... ON).

- Dữ liệu sai lệch thực tế: Tạo ra các tổ hợp dư thừa và không có thật. Ví dụ: Nguyễn Minh Hoàng thực tế chỉ học môn 12 nhưng trong bảng vẫn bị ghép thêm với môn 26 và 34.

- Xử lý giá trị rỗng (NaN): Các sinh viên chưa có mã môn học (như Phạm Văn Nam) vẫn bị đem đi ghép với cả 3 môn học.

=> Kết luận: Tích Descartes chỉ tạo ra tất cả các tổ hợp có thể, sinh ra nhiều "dữ liệu rác" và không phản ánh đúng việc sinh viên nào đang học môn nào trong thực tế.

## SỬ DỤNG JOIN: INNER JOIN
Chỉ giữ lại các dòng có `student.course_id = course.id`.

In [50]:
show(
    "INNER JOIN",
    '''
    SELECT
        s.student_id,
        s.name,
        s.class,
        s.course_id,
        s.score,
        c.course_name
    FROM student AS s
    INNER JOIN course AS c
        ON s.course_id = c.id
    ORDER BY s.student_id;
    '''
)

### INNER JOIN

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke


### Kết luận về INNER JOIN:

- Số lượng kết quả: Bảng chỉ còn lại đúng 3 dòng (so với 10 sinh viên ban đầu).

- Bản chất INNER JOIN: Phép nối này chỉ giữ lại những bản ghi có mã course_id khớp chính xác ở cả hai bảng student và course.

Dữ liệu bị loại bỏ: Những sinh viên chưa có môn (NaN), sai mã môn (20, 24) và những môn học không có ai đăng ký (mã 26) đều bị loại bỏ hoàn toàn.

=> Kết luận: INNER JOIN hoạt động như một bộ lọc nghiêm ngặt, chỉ lấy ra những sinh viên đang học các môn có thực trong danh sách môn học đã khai báo.

## LEFT JOIN

In [51]:
show(
    "LEFT JOIN",
    '''
    SELECT
        s.student_id,
        s.name,
        s.class,
        s.course_id,
        s.score,
        c.course_name
    FROM student AS s
    LEFT JOIN course AS c
        ON s.course_id = c.id
    ORDER BY s.student_id;
    '''
)

### LEFT JOIN

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,None


### KẾT LUẬN VỀ LEFT JOIN:

- Số lượng kết quả: Trả về đủ 10 dòng (giữ nguyên toàn bộ danh sách 10 sinh viên ban đầu).

- Nguyên lý: Ưu tiên bảng bên trái (student). Dù có tìm thấy môn học tương ứng trong bảng course hay không, sinh viên đó vẫn xuất hiện.

- Xử lý dữ liệu không khớp: Đối với những sinh viên chưa có mã môn học (NaN) hoặc mã môn bị sai (20, 24), hệ thống vẫn giữ thông tin sinh viên nhưng điền None (NULL) vào cột course_name.

=> Kết luận: LEFT JOIN rất hữu ích để xem toàn bộ danh sách sinh viên, đồng thời phát hiện ra ngay những ai đang bị lỗi mã môn hoặc chưa đăng ký môn học (nhìn vào các ô None).

## RIGHT JOIN
SQLite thường không hỗ trợ `RIGHT JOIN` theo cách cổ điển, nên notebook này dùng **LEFT JOIN đảo chiều** để mô phỏng `RIGHT JOIN`.
Giữ toàn bộ dữ liệu bảng `course`.

In [52]:
show(
    "RIGHT JOIN (mô phỏng bằng LEFT JOIN đảo chiều)",
    '''
    SELECT
        s.student_id,
        s.name,
        s.class,
        s.course_id,
        s.score,
        c.id AS course_id_from_course,
        c.course_name
    FROM course AS c
    LEFT JOIN student AS s
        ON s.course_id = c.id
    ORDER BY c.id, s.student_id;
    '''
)

### RIGHT JOIN (mô phỏng bằng LEFT JOIN đảo chiều)

,student_id,name,class,course_id,score,course_id_from_course,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,NaN,None,None,NaN,NaN,26,Tin hoc
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
3,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34,Thong ke


### Kết luận về right join:

Kết quả: Trả về 4 dòng.

Nguyên lý: Ưu tiên bảng course (bảng bên phải theo logic, hoặc đưa lên làm bảng bên trái trong code). Hệ thống giữ lại toàn bộ danh sách môn học.

Xử lý dữ liệu không khớp: Môn "Tin học" (mã 26) không có sinh viên nào đăng ký, nên toàn bộ thông tin về sinh viên đều trống (NaN/None). Các sinh viên chưa đăng ký môn hoặc đăng ký sai mã môn (mã 20, 24) đều bị loại bỏ hoàn toàn khỏi kết quả.

Kết luận: Phép kết nối này giúp tập trung vào danh sách môn học. Nó rất hữu ích để xem mỗi môn đang có ai học và dễ dàng phát hiện ngay môn nào đang không có sinh viên nào đăng ký

## FULL OUTER JOIN


In [53]:
show(
    "FULL OUTER JOIN",
    '''
    SELECT
        s.student_id,
        s.name,
        s.class,
        s.course_id AS student_course_id,
        s.score,
        c.id AS course_id_from_course,
        c.course_name
    FROM student AS s
    LEFT JOIN course AS c
        ON s.course_id = c.id

    UNION ALL

    SELECT
        s.student_id,
        s.name,
        s.class,
        s.course_id AS student_course_id,
        s.score,
        c.id AS course_id_from_course,
        c.course_name
    FROM course AS c
    LEFT JOIN student AS s
        ON s.course_id = c.id
    WHERE s.student_id IS NULL
    ORDER BY student_id, course_id_from_course;
    '''
)

### FULL OUTER JOIN

,student_id,name,class,student_course_id,score,course_id_from_course,course_name
0,NaN,None,None,NaN,NaN,26.0,Tin hoc
1,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
3,3.0,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
4,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
5,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
6,6.0,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
7,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
8,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
9,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None


###Kết luận FULL OUTER JOIN:

Kết quả: Trả về tổng cộng 11 dòng.

Nguyên lý: Là sự gộp chung của cả LEFT JOIN và RIGHT JOIN. Nó giữ lại toàn bộ bản ghi từ cả hai bảng, bất kể có tìm thấy dữ liệu khớp hay không.

Xử lý dữ liệu không khớp: Thể hiện sự thiếu sót từ cả hai phía: Giữ lại tất cả các sinh viên bị sai mã môn hoặc chưa đăng ký môn (thông tin môn học sẽ là NaN/None). Giữ lại cả môn "Tin học" (mã 26) dù không có sinh viên nào đăng ký (thông tin sinh viên sẽ là NaN/None ở dòng số 0).

=> Kết luận: FULL OUTER JOIN mang lại bức tranh toàn cảnh nhất. Nó giúp bạn đối chiếu toàn diện hai bảng để thấy ngay những điểm đã khớp, những sinh viên đang bị không có môn và những môn học đang bị không có sinh viên

# **2.Cập nhật course_id bị thiếu và loại bỏ bản ghi không hợp lệ**

In [54]:
run_script('''
DROP TABLE IF EXISTS student_work;
CREATE TABLE student_work AS
SELECT * FROM student;
''')

show(
    "Các bản ghi thiếu course_id hoặc course_id không tồn tại trong bảng course",
    '''
    SELECT *
    FROM student_work
    WHERE course_id IS NULL
       OR course_id NOT IN (SELECT id FROM course)
    ORDER BY student_id;
    '''
)

### Các bản ghi thiếu course_id hoặc course_id không tồn tại trong bảng course

,student_id,name,class,course_id,score
0,3,Pham Van Nam,Toan Tin,NaN,7.9
1,4,Le Thanh Huyen,Toan Tin,20.0,7.2
2,5,Vu Quoc Anh,May Tinh,24.0,8.0
3,6,Dang Thuy Linh,May Tinh,24.0,5.5
4,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
5,9,Duong Huu Phuc,Kinh Te,NaN,7.2
6,10,Cao Thi Hanh,May Tinh,NaN,7.0


In [55]:
#Điền course_id còn thiếu
run_script('''
UPDATE student_work
SET course_id = CASE
    WHEN course_id IS NULL AND class = 'Toan Tin' THEN 12
    WHEN course_id IS NULL AND class = 'Kinh Te'  THEN 34
    WHEN course_id IS NULL AND class = 'May Tinh' THEN 26
    ELSE course_id
END
WHERE course_id IS NULL;
''')

show("Bảng student_work sau khi điền course_id còn thiếu", "SELECT * FROM student_work ORDER BY student_id;")

### Bảng student_work sau khi điền course_id còn thiếu

,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,12,7.9
3,4,Le Thanh Huyen,Toan Tin,20,7.2
4,5,Vu Quoc Anh,May Tinh,24,8.0
5,6,Dang Thuy Linh,May Tinh,24,5.5
6,7,Bui Tien Dung,Kinh Te,34,9.2
7,8,Ho Ngoc Mai,Toan Tin,20,8.8
8,9,Duong Huu Phuc,Kinh Te,34,7.2
9,10,Cao Thi Hanh,May Tinh,26,7.0


In [56]:
#Loại bỏ các bản ghi tham gia môn học không tồn tại (Các course_id không hợp lệ là: 20, 24.)
run_script('''
DELETE FROM student_work
WHERE course_id NOT IN (SELECT id FROM course);
''')

show(
    "student_work sau khi loại bỏ course_id không hợp lệ",
    '''
    SELECT s.student_id, s.name, s.class, s.course_id, c.course_name, s.score
    FROM student_work s
    LEFT JOIN course c ON s.course_id = c.id
    ORDER BY s.student_id;
    '''
)

### student_work sau khi loại bỏ course_id không hợp lệ

,student_id,name,class,course_id,course_name,score
0,1,Nguyen Minh Hoang,May Tinh,12,Giai tich,6.7
1,2,Tran Thi Lan,Kinh Te,34,Thong ke,9.2
2,3,Pham Van Nam,Toan Tin,12,Giai tich,7.9
3,7,Bui Tien Dung,Kinh Te,34,Thong ke,9.2
4,9,Duong Huu Phuc,Kinh Te,34,Thong ke,7.2
5,10,Cao Thi Hanh,May Tinh,26,Tin hoc,7.0


### KẾT LUẬN:

Mục tiêu: Chuẩn hóa và dọn dẹp dữ liệu lỗi.

Thao tác: Xử lý thông minh bằng cách điền mã môn bị thiếu dựa theo logic của từng lớp học, sau đó dứt khoát loại bỏ các bản ghi chứa mã môn "ảo" (20, 24).

Kết quả cuối cùng: Dữ liệu đã sạch 100%. Từ 10 bản ghi lộn xộn ban đầu, bảng đã được chắt lọc còn 6 sinh viên hợp lệ. Đảm bảo tính toàn vẹn tham chiếu: mọi sinh viên đều có môn học khớp hoàn toàn với bảng course.

## **2.a. Tổng số sinh viên và điểm trung bình của từng lớp**

In [57]:
show(
    "Tổng số sinh viên và điểm trung bình theo lớp",
    '''
    SELECT
        class,
        COUNT(*) AS total_students,
        ROUND(AVG(score), 2) AS avg_score
    FROM student_work
    GROUP BY class
    ORDER BY class;
    '''
)

### Tổng số sinh viên và điểm trung bình theo lớp

,class,total_students,avg_score
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


### Nhận xét:

Đánh giá dữ liệu: Nhìn vào bảng, lớp "Kinh Tế" đang dẫn đầu cả về số lượng (3 sinh viên) lẫn chất lượng (điểm trung bình cao nhất: 8.53). Ngược lại, lớp "Toán Tin" hao hụt nhiều nhất, chỉ còn duy nhất 1 sinh viên hợp lệ.

## **2.b. Tổng số sinh viên và điểm trung bình của từng môn học**

In [58]:
show(
    "Tổng số sinh viên và điểm trung bình theo môn học",
    '''
    SELECT
        c.id AS course_id,
        c.course_name,
        COUNT(s.student_id) AS total_students,
        ROUND(AVG(s.score), 2) AS avg_score
    FROM course c
    LEFT JOIN student_work s
        ON s.course_id = c.id
    GROUP BY c.id, c.course_name
    ORDER BY c.id;
    '''
)

### Tổng số sinh viên và điểm trung bình theo môn học

,course_id,course_name,total_students,avg_score
0,12,Giai tich,2,7.30
1,26,Tin hoc,1,7.00
2,34,Thong ke,3,8.53


### Nhận xét:

Tính chính xác: Tổng số sinh viên của 3 môn cộng lại đúng bằng 6. Điều này chứng tỏ code đã thống kê chuẩn xác trên tệp dữ liệu đã được làm sạch ở bước trước đó.

Đánh giá dữ liệu:
- Môn "Thống kê" đang dẫn đầu cả về số lượng người học (3 sinh viên) và điểm trung bình (8.53).

- Môn "Tin học" ban đầu không có ai đăng ký, nhưng nhờ quá trình khôi phục dữ liệu lỗi (điền mã môn dựa trên lớp học ở bước trên), hệ thống đã ghi nhận 1 sinh viên tham gia môn này.

## **2.c. Phân loại thi đua theo điểm trung bình của từng môn học**

Quy tắc:
- Điểm TB ≥ 9.0 → Xuất sắc
- 6.0 ≤ Điểm TB ≤ 8.9 → Tốt
- Điểm TB < 6.0 → Kém

In [59]:
show(
    "Phân loại thi đua theo môn học",
    '''
    WITH course_avg AS (
        SELECT
            c.id AS course_id,
            c.course_name,
            ROUND(AVG(s.score), 2) AS avg_score
        FROM course c
        LEFT JOIN student_work s
            ON s.course_id = c.id
        GROUP BY c.id, c.course_name
    )
    SELECT
        course_id,
        course_name,
        avg_score,
        CASE
            WHEN avg_score >= 9.0 THEN 'Xuat sac'
            WHEN avg_score >= 6.0 AND avg_score <= 8.9 THEN 'Tot'
            WHEN avg_score < 6.0 THEN 'Kem'
            ELSE 'Khong co du lieu'
        END AS classification
    FROM course_avg
    ORDER BY course_id;
    '''
)

### Phân loại thi đua theo môn học

,course_id,course_name,avg_score,classification
0,12,Giai tich,7.30,Tot
1,26,Tin hoc,7.00,Tot
2,34,Thong ke,8.53,Tot


### Nhận xét:

Tính chính xác: Code phân loại sử dụng mệnh đề CASE WHEN đã hoạt động chính xác dựa trên điểm trung bình (avg_score) đã tính toán. Cả ba môn "Giải tích" (7.30), "Tin học" (7.00), và "Thống kê" (8.53) đều rơi vào khoảng từ 6.0 đến 8.9, do đó đều được xếp loại 'Tot'.

Đánh giá tổng quan: Không có môn học nào đạt mức 'Xuat sac' ($\ge$ 9.0) hay bị xếp loại 'Kem' (< 6.0) dựa trên kết quả trung bình. Điều này cho thấy chất lượng điểm số khá đồng đều ở mức khá/tốt sau khi đã làm sạch dữ liệu.

# **3.Xếp hạng sinh viên**


## **3.a. Xếp hạng theo điểm số toàn bộ sinh viên**

In [60]:
show(
    "Bảng xếp hạng toàn bộ sinh viên theo điểm giảm dần",
    '''
    SELECT
        student_id,
        name,
        class,
        course_id,
        score,
        RANK() OVER (ORDER BY score DESC) AS rank_desc
    FROM student_work
    ORDER BY rank_desc, student_id;
    '''
)

show(
    "Top 3 thứ hạng cao nhất toàn bộ sinh viên",
    '''
    WITH ranked AS (
        SELECT
            student_id,
            name,
            class,
            course_id,
            score,
            RANK() OVER (ORDER BY score DESC) AS rank_desc
        FROM student_work
    )
    SELECT *
    FROM ranked
    WHERE rank_desc <= 3
    ORDER BY rank_desc, student_id;
    '''
)

show(
    "Top 3 thứ hạng thấp nhất toàn bộ sinh viên",
    '''
    WITH ranked AS (
        SELECT
            student_id,
            name,
            class,
            course_id,
            score,
            RANK() OVER (ORDER BY score ASC) AS rank_asc
        FROM student_work
    )
    SELECT *
    FROM ranked
    WHERE rank_asc <= 3
    ORDER BY rank_asc, student_id;
    '''
)

### Bảng xếp hạng toàn bộ sinh viên theo điểm giảm dần

,student_id,name,class,course_id,score,rank_desc
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,12,7.9,3
3,9,Duong Huu Phuc,Kinh Te,34,7.2,4
4,10,Cao Thi Hanh,May Tinh,26,7.0,5
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6


### Top 3 thứ hạng cao nhất toàn bộ sinh viên

,student_id,name,class,course_id,score,rank_desc
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,12,7.9,3


### Top 3 thứ hạng thấp nhất toàn bộ sinh viên

,student_id,name,class,course_id,score,rank_asc
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,26,7.0,2
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3


### NHẬN XÉT:

Xử lý đồng hạng (Top 3 cao nhất): Hàm RANK() đã thể hiện rõ đặc tính bỏ qua thứ hạng khi có người bằng điểm. Trần Thị Lan và Bùi Tiến Dũng cùng đạt 9.2 điểm nên cùng giữ hạng 1, đẩy Phạm Văn Nam (7.9 điểm) xuống vị trí hạng 3. Phép lọc lấy đúng 3 gương mặt xuất sắc nhất.

Top 3 thấp nhất: Kỹ thuật đảo chiều sắp xếp ORDER BY score ASC đã hoạt động hiệu quả, gọi tên chính xác 3 sinh viên có điểm số "đội sổ" (từ 6.7 đến 7.2 điểm). Mức điểm thấp nhất ghi nhận là 6.7, cho thấy chất lượng lớp vẫn ở mức khá tốt.

## **3.b Xếp hạng theo lớp học**

In [61]:
show(
    "Bảng xếp hạng theo lớp (điểm giảm dần)",
    '''
    SELECT
        student_id,
        name,
        class,
        course_id,
        score,
        RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
    FROM student_work
    ORDER BY class, rank_in_class, student_id;
    '''
)

show(
    "Top 3 thứ hạng cao nhất trong từng lớp",
    '''
    WITH ranked AS (
        SELECT
            student_id,
            name,
            class,
            course_id,
            score,
            RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
        FROM student_work
    )
    SELECT *
    FROM ranked
    WHERE rank_in_class <= 3
    ORDER BY class, rank_in_class, student_id;
    '''
)

show(
    "Top 3 thứ hạng thấp nhất trong từng lớp",
    '''
    WITH ranked AS (
        SELECT
            student_id,
            name,
            class,
            course_id,
            score,
            RANK() OVER (PARTITION BY class ORDER BY score ASC) AS rank_in_class_low
        FROM student_work
    )
    SELECT *
    FROM ranked
    WHERE rank_in_class_low <= 3
    ORDER BY class, rank_in_class_low, student_id;
    '''
)

### Bảng xếp hạng theo lớp (điểm giảm dần)

,student_id,name,class,course_id,score,rank_in_class
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3
3,10,Cao Thi Hanh,May Tinh,26,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,12,7.9,1


### Top 3 thứ hạng cao nhất trong từng lớp

,student_id,name,class,course_id,score,rank_in_class
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3
3,10,Cao Thi Hanh,May Tinh,26,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,12,7.9,1


### Top 3 thứ hạng thấp nhất trong từng lớp

,student_id,name,class,course_id,score,rank_in_class_low
0,9,Duong Huu Phuc,Kinh Te,34,7.2,1
1,2,Tran Thi Lan,Kinh Te,34,9.2,2
2,7,Bui Tien Dung,Kinh Te,34,9.2,2
3,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
4,10,Cao Thi Hanh,May Tinh,26,7.0,2
5,3,Pham Van Nam,Toan Tin,12,7.9,1


### NHẬN XÉT:

Hiện tượng "Top 3 lấy tất cả": Có thể nhận thấy cả bảng Top 3 cao nhất và Top 3 thấp nhất đều trả về đủ 6 sinh viên của toàn trường. Lý do rất đơn giản: sau khi làm sạch dữ liệu ở bước trước, lớp đông nhất là "Kinh Tế" cũng chỉ còn đúng 3 thành viên, các lớp khác có 1-2 người. Do đó, điều kiện lọc rank_in_class <= 3 mặc nhiên đúng với tất cả mọi người trong lớp.

Tính chính xác khi đảo chiều: Ở bảng Top 3 thấp nhất của lớp "Kinh Tế", hệ thống ghi nhận Dương Hữu Phúc (7.2 điểm) xếp hạng 1 "từ dưới lên", còn Trần Thị Lan và Bùi Tiến Dũng (cùng 9.2 điểm) đồng hạng 2. Điều này chứng tỏ hàm cửa sổ đã xử lý hoàn hảo việc sắp xếp ngược (ORDER BY score ASC) và đánh giá đồng hạng ngay bên trong các phân nhóm nhỏ.

## **3c. Xếp hạng theo mã môn học**

In [62]:
show(
    "Bảng xếp hạng theo mã môn học (điểm giảm dần)",
    '''
    SELECT
        student_id,
        name,
        class,
        course_id,
        score,
        RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_in_course
    FROM student_work
    ORDER BY course_id, rank_in_course, student_id;
    '''
)

show(
    "Top 3 thứ hạng cao nhất trong từng môn học",
    '''
    WITH ranked AS (
        SELECT
            student_id,
            name,
            class,
            course_id,
            score,
            RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_in_course
        FROM student_work
    )
    SELECT *
    FROM ranked
    WHERE rank_in_course <= 3
    ORDER BY course_id, rank_in_course, student_id;
    '''
)

show(
    "Top 3 thứ hạng thấp nhất trong từng môn học",
    '''
    WITH ranked AS (
        SELECT
            student_id,
            name,
            class,
            course_id,
            score,
            RANK() OVER (PARTITION BY course_id ORDER BY score ASC) AS rank_in_course_low
        FROM student_work
    )
    SELECT *
    FROM ranked
    WHERE rank_in_course_low <= 3
    ORDER BY course_id, rank_in_course_low, student_id;
    '''
)

### Bảng xếp hạng theo mã môn học (điểm giảm dần)

,student_id,name,class,course_id,score,rank_in_course
0,3,Pham Van Nam,Toan Tin,12,7.9,1
1,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
2,10,Cao Thi Hanh,May Tinh,26,7.0,1
3,2,Tran Thi Lan,Kinh Te,34,9.2,1
4,7,Bui Tien Dung,Kinh Te,34,9.2,1
5,9,Duong Huu Phuc,Kinh Te,34,7.2,3


### Top 3 thứ hạng cao nhất trong từng môn học

,student_id,name,class,course_id,score,rank_in_course
0,3,Pham Van Nam,Toan Tin,12,7.9,1
1,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
2,10,Cao Thi Hanh,May Tinh,26,7.0,1
3,2,Tran Thi Lan,Kinh Te,34,9.2,1
4,7,Bui Tien Dung,Kinh Te,34,9.2,1
5,9,Duong Huu Phuc,Kinh Te,34,7.2,3


### Top 3 thứ hạng thấp nhất trong từng môn học

,student_id,name,class,course_id,score,rank_in_course_low
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,3,Pham Van Nam,Toan Tin,12,7.9,2
2,10,Cao Thi Hanh,May Tinh,26,7.0,1
3,9,Duong Huu Phuc,Kinh Te,34,7.2,1
4,2,Tran Thi Lan,Kinh Te,34,9.2,2
5,7,Bui Tien Dung,Kinh Te,34,9.2,2


### Nhận xét:

Hiện tượng gom trọn danh sách: Tương tự như khi xếp hạng theo lớp, vì môn học đông nhất (Thống kê - mã 34) hiện tại cũng chỉ có đúng 3 sinh viên đăng ký, nên điều kiện lọc Top 3 (cả cao nhất lẫn thấp nhất) mặc nhiên bao gồm toàn bộ 6 sinh viên hợp lệ của cả trường.

Độ chính xác của hàm RANK: Hàm cửa sổ tiếp tục xử lý mượt mà các ca đồng hạng. Ở môn Thống kê (mã 34), Trần Thị Lan và Bùi Tiến Dũng (cùng 9.2 điểm) cùng giữ hạng 1 khi xét điểm cao nhất, và lùi xuống đồng hạng 2 một cách logic khi hệ thống đảo chiều tìm điểm thấp nhất (ORDER BY score ASC).

# **4.Bổ sung trường graduation_date**

Giả định ở đây: thời gian tốt nghiệp được tính bằng thời gian hiện tại + số ngày bằng thứ hạng toàn cục theo điểm số.

Ví dụ:
- hạng 1 → cộng 1 ngày
- hạng 3 → cộng 3 ngày
- hạng 6 → cộng 6 ngày

In [63]:
run_script('''
ALTER TABLE student_work ADD COLUMN graduation_date DATETIME;
''')

run_script('''
WITH ranked AS (
    SELECT
        student_id,
        RANK() OVER (ORDER BY score DESC) AS overall_rank
    FROM student_work
)
UPDATE student_work
SET graduation_date = (
    SELECT datetime('now', 'localtime', '+' || overall_rank || ' days')
    FROM ranked
    WHERE ranked.student_id = student_work.student_id
);
''')

show(
    "Bảng student_work sau khi thêm graduation_date",
    '''
    SELECT
        student_id,
        name,
        class,
        course_id,
        score,
        graduation_date
    FROM student_work
    ORDER BY score DESC, student_id;
    '''
)

### Bảng student_work sau khi thêm graduation_date

,student_id,name,class,course_id,score,graduation_date
0,2,Tran Thi Lan,Kinh Te,34,9.2,2026-04-02 16:16:06
1,7,Bui Tien Dung,Kinh Te,34,9.2,2026-04-02 16:16:06
2,3,Pham Van Nam,Toan Tin,12,7.9,2026-04-04 16:16:06
3,9,Duong Huu Phuc,Kinh Te,34,7.2,2026-04-05 16:16:06
4,10,Cao Thi Hanh,May Tinh,26,7.0,2026-04-06 16:16:06
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,2026-04-07 16:16:06


### NHẬN XÉT:
- Sự phân hóa ngày tốt nghiệp:
    - Các sinh viên đồng hạng 1 (Trần Thị Lan và Bùi Tiến Dũng với 9.2 điểm) có cùng ngày tốt nghiệp (được cộng thêm 1 ngày so với thời điểm chạy code).
    - Sinh viên có điểm thấp nhất là Nguyễn Minh Hoàng có ngày tốt nghiệp xa nhất.
- Cột graduation_date được lưu trữ dưới định dạng DATETIME, đảm bảo tính chuẩn hóa để có thể sử dụng cho các phép toán thời gian hoặc lọc dữ liệu theo lịch trình trong tương lai.
